# 🗄️ Week 3 Assignment — Company Database

**Course:** Python Programming — Five-Week Foundations  
**Week:** 3 — SQLite · CREATE TABLE · INSERT · SELECT · UPDATE · DELETE · CSV Import  
**Estimated time:** 2–3 hours  
**Total points:** 100  

---

## Scenario

You have been given `company.db` — a SQLite database that already contains two tables: `departments` and `employees`. Your job is to connect to it, explore its structure, practise all four CRUD operations (Create, Read, Update, Delete), run analytical queries, and finally import your cleaned CSV from Week 2.

**Files in this folder:**
| File | Description |
|------|-------------|
| `week3_assignment.ipynb` | This notebook |
| `company.db` | Starter SQLite database |
| `employees_clean.csv` | Your cleaned output from Week 2 (copy it here) |

> ⚠️ **Before you start:** activate your virtual environment — `source venv/bin/activate` — and make sure `company.db` is in the same folder as this notebook.

---

## Reference — Connecting to SQLite

Python's built-in `sqlite3` module needs no installation. Every interaction follows the same pattern:

```python
import sqlite3

conn   = sqlite3.connect("company.db")   # open (or create) the database file
cursor = conn.cursor()                   # cursor sends SQL commands

cursor.execute("SELECT * FROM employees")
rows = cursor.fetchall()                 # retrieve results

conn.commit()   # save any INSERT / UPDATE / DELETE changes
conn.close()    # always close when done
```

### Using `with` for automatic cleanup

Just like `with open()` for files, you can use `with sqlite3.connect()` and the connection closes automatically:

```python
with sqlite3.connect("company.db") as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM employees")
    print(cursor.fetchone()[0])
# connection is closed here automatically
```

### Row factory — getting dictionaries instead of tuples

By default, each row comes back as a plain tuple: `(1, 'Alice', 31, ...)`. Setting `row_factory` lets you access columns by name instead:

```python
conn = sqlite3.connect("company.db")
conn.row_factory = sqlite3.Row          # enable named access

cursor = conn.cursor()
cursor.execute("SELECT * FROM employees WHERE city = ?", ("Chicago",))

for row in cursor.fetchall():
    print(row["name"], row["salary"])   # column name as key
```

> 💡 Always set `row_factory = sqlite3.Row` — it makes your code much easier to read and avoids errors from counting tuple positions.

---

## Reference — Core SQL Statements

### SELECT — reading data

```python
# All columns, all rows
cursor.execute("SELECT * FROM employees")

# Specific columns
cursor.execute("SELECT name, salary FROM employees")

# Filter with WHERE
cursor.execute("SELECT * FROM employees WHERE city = ?", ("Chicago",))

# Sort results  (ASC = low→high, DESC = high→low)
cursor.execute("SELECT * FROM employees ORDER BY salary DESC")

# Limit the number of rows returned
cursor.execute("SELECT * FROM employees ORDER BY salary DESC LIMIT 5")

# Aggregate functions
cursor.execute("SELECT COUNT(*), AVG(salary), MIN(salary), MAX(salary) FROM employees")
```

### Fetching results

```python
cursor.execute("SELECT * FROM employees")

rows = cursor.fetchall()    # list of all matching rows
row  = cursor.fetchone()    # only the first row (or None if no results)
```

### INSERT — adding rows

```python
# Single row — always use ? placeholders, never f-strings
cursor.execute(
    "INSERT INTO employees (name, age, city, salary) VALUES (?, ?, ?, ?)",
    ("Alice", 30, "New York", 75000.0)
)

# Multiple rows at once
new_employees = [
    ("Bob",   25, "Chicago",  62000.0),
    ("Carol", 35, "Boston",   88000.0),
]
cursor.executemany(
    "INSERT INTO employees (name, age, city, salary) VALUES (?, ?, ?, ?)",
    new_employees
)
conn.commit()   # required to save inserts
```

### UPDATE — changing existing rows

```python
# Update one employee's salary
cursor.execute(
    "UPDATE employees SET salary = ? WHERE name = ?",
    (80000.0, "Alice")
)

# Give everyone in a city a 5% raise
cursor.execute(
    "UPDATE employees SET salary = salary * 1.05 WHERE city = ?",
    ("New York",)
)
conn.commit()
```

### DELETE — removing rows

```python
cursor.execute("DELETE FROM employees WHERE name = ?", ("Dave",))
conn.commit()
```

> ⚠️ **Always call `conn.commit()` after INSERT, UPDATE, or DELETE.** Without it, your changes exist only in memory and are lost when the connection closes.

---

## Reference — Parameterized Queries and SQL Injection

**Never** build SQL strings by concatenating variables or using f-strings. This creates a security vulnerability called SQL injection — a malicious value could modify or destroy your database.

```python
# ❌ UNSAFE — never do this
city = "Chicago"
cursor.execute(f"SELECT * FROM employees WHERE city = '{city}'")

# ✅ SAFE — always use ? placeholders
cursor.execute("SELECT * FROM employees WHERE city = ?", (city,))
```

The `?` placeholders tell SQLite to treat the value as data, never as SQL code. Notice the value is always passed as a **tuple** — even for a single value, you need the trailing comma: `(city,)` not `(city)`.

```python
# Single value — tuple needs a trailing comma
cursor.execute("SELECT * FROM employees WHERE id = ?", (5,))     # ✅ correct
cursor.execute("SELECT * FROM employees WHERE id = ?", (5))      # ❌ wrong — (5) is just the number 5

# Multiple values — no trailing comma needed
cursor.execute("SELECT * FROM employees WHERE city = ? AND salary > ?", ("Austin", 80000))
```

---

## Task 1 — Explore the Database
**15 points**

Connect to `company.db` and find out what's inside before writing any queries.

**Requirements:**
1. Connect to `company.db` and set `row_factory = sqlite3.Row`.
2. Print the names of all tables in the database.  
   > Hint: `SELECT name FROM sqlite_master WHERE type='table'` lists all tables.
3. Print all rows from the `departments` table.
4. Print the total number of rows in the `employees` table.
5. Print the column names of the `employees` table.  
   > Hint: `PRAGMA table_info(employees)` returns one row per column; each row has a `name` field.

In [1]:
# Task 1 — Explore the database
import sqlite3

conn = sqlite3.connect("company.db")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

# 2. List all tables
print("--- Tables ---")
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
for row in tables:
    print(" ", row["name"])

# 3. All departments
print("\n--- Departments ---")
cursor.execute("SELECT * FROM departments")
departments = cursor.fetchall()
for row in departments:
    print(dict(row))

# 4. Total employee count
print("\n--- Employee Count ---")
cursor.execute("SELECT COUNT(*) AS count FROM employees")
employee_count = cursor.fetchone()
print(employee_count["count"])

# 5. Column names of employees table
print("\n--- Employee Columns ---")
cursor.execute("PRAGMA table_info(employees)")
columns = cursor.fetchall()
for column in columns:
    print(" ", column["name"])


conn.close()

--- Tables ---
  departments
  sqlite_sequence
  employees
  employees_imported

--- Departments ---
{'id': 1, 'name': 'Engineering'}
{'id': 2, 'name': 'Marketing'}
{'id': 3, 'name': 'Sales'}
{'id': 4, 'name': 'HR'}

--- Employee Count ---
25

--- Employee Columns ---
  id
  name
  age
  department
  city
  salary
  start_date
  email


---
## Task 2 — SELECT Queries
**20 points**

Practise reading data from the `employees` table using a variety of SELECT queries.

**Requirements:**
1. Print every employee's `name` and `salary`, formatted as: `Alice Nguyen        $95,000.00`
2. Print all employees in the `Engineering` department, ordered by salary descending.
3. Print all employees earning more than `$80,000`, showing name, department, and salary.
4. Print the `COUNT`, `AVG`, `MIN`, and `MAX` salary across all employees.
5. Print all employees located in `Chicago` or `Austin` — use a single query with `WHERE city IN (?, ?)` syntax.

> 💡 **Hint:** `f"{row['name']:<20}"` left-aligns a string in a field 20 characters wide — useful for neat column output.

In [2]:
# Task 2 — SELECT queries
conn = sqlite3.connect("company.db")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

# 1. All employees — name and salary
print("--- All Employees ---")
cursor.execute("SELECT name, salary FROM employees")
employees = cursor.fetchall()
for employee in employees:
    print(dict(employee))

# 2. Engineering department, sorted by salary descending
print("\n--- Engineering (by salary) ---")
cursor.execute('SELECT * FROM employees WHERE department = ? ORDER BY salary DESC',
                ("Engineering",))
for employee in cursor.fetchall():
    print(dict(employee))

# 3. Employees earning over $80,000
print("\n--- Earning over $80,000 ---")
cursor.execute('SELECT * FROM employees WHERE salary > ?',
                (80000,))
high_salary_employees = cursor.fetchall()
for employee in high_salary_employees:
    print(dict(employee))


# 4. Salary statistics
print("\n--- Salary Statistics ---")
cursor.execute('SELECT MIN(salary) AS min_salary, MAX(salary) AS max_salary, AVG(salary) AS avg_salary FROM employees')
salary_stats = cursor.fetchone()
print(dict(salary_stats))

# 5. Employees in Chicago or Austin
print("\n--- Chicago and Austin ---")
cursor.execute('SELECT * FROM employees WHERE city IN (?, ?)',
               ('Chicago', 'Austin'))
city_employees = cursor.fetchall()
for employee in city_employees:
    print(dict(employee))
conn.close()

--- All Employees ---
{'name': 'Alice Nguyen', 'salary': 107100.0}
{'name': 'Bob Martinez', 'salary': 62000.0}
{'name': 'Carol Kim', 'salary': 123480.0}
{'name': 'Dave Okafor', 'salary': 71000.0}
{'name': 'Eve Patel', 'salary': 58000.0}
{'name': 'Frank Li', 'salary': 84000.0}
{'name': 'Grace Thompson', 'salary': 113557.5}
{'name': 'Hana Sato', 'salary': 47000.0}
{'name': 'Ivan Rossi', 'salary': 108045.0}
{'name': 'Julia Fernandez', 'salary': 91000.0}
{'name': "Kevin O'Brien", 'salary': 67500.0}
{'name': 'Laura Chen', 'salary': 61000.0}
{'name': 'Marcus Reed', 'salary': 131197.5}
{'name': 'Nina Johansson', 'salary': 109147.5}
{'name': 'Omar Hassan', 'salary': 52000.0}
{'name': 'Paula Diaz', 'salary': 87500.0}
{'name': 'Quinn Baker', 'salary': 66000.0}
{'name': 'Rachel Wong', 'salary': 117967.5}
{'name': 'Sam Nkosi', 'salary': 59000.0}
{'name': 'Uma Sharma', 'salary': 70000.0}
{'name': 'Victor Alves', 'salary': 137812.5}
{'name': 'Wendy Park', 'salary': 48500.0}
{'name': 'Xavier Dubois',

---
## Task 3 — INSERT New Employees
**15 points**

Add new employees to the database using both single and bulk insert.

**Requirements:**
1. Insert one new employee using a single `cursor.execute()` with `?` placeholders. Use any realistic values you like.
2. Insert three more employees at once using `cursor.executemany()`.
3. After inserting, print the total employee count to confirm the new rows were added.
4. Print the `id` and `name` of the last 5 employees in the table (ordered by `id DESC LIMIT 5`) to verify your inserts.

> 💡 **Hint:** After inserting, `cursor.lastrowid` gives you the `id` that was auto-assigned to the most recent single insert — useful for confirming it worked.

In [3]:
# Task 3 — INSERT new employees
conn = sqlite3.connect("company.db")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

# 1. Single insert
cursor.execute(
    "INSERT INTO employees (name, age, department, city, salary, start_date, email) VALUES (?, ?, ?, ?, ?, ?, ?)",
    ("Chad Palumbo", 23, "Marketing", "New York", 110000, "2026-01-15", "chad.palumbo@princeton.edu")
)
print(f"Inserted employee with id: {cursor.lastrowid}")

# 2. Bulk insert with executemany
new_employees = [
    ("Alice Johnson", 28, "Engineering", "Chicago", 95000, "2023-06-01", "alice.johnson@princeton.edu"),
    ("Bob Smith", 35, "Sales", "Austin", 85000, "2023-07-15", "bob.smith@princeton.edu"),
    ("Charlie Brown", 42, "Human Resources", "New York", 90000, "2023-08-20", "charlie.brown@princeton.edu"),
]
cursor.executemany(
    "INSERT INTO employees (name, age, department, city, salary, start_date, email) VALUES (?, ?, ?, ?, ?, ?, ?)",
    new_employees
)

conn.commit()

# 3. Confirm total count
cursor.execute("SELECT COUNT(*) AS total FROM employees")
print(f"Total employees now: {cursor.fetchone()['total']}")

# 4. Show the last 5 rows
print("\n--- Last 5 Inserts ---")
cursor.execute("SELECT * FROM employees ORDER BY id DESC LIMIT 5")
for employee in cursor.fetchall():
    print(dict(employee))
conn.close()

Inserted employee with id: 34
Total employees now: 29

--- Last 5 Inserts ---
{'id': 37, 'name': 'Charlie Brown', 'age': 42, 'department': 'Human Resources', 'city': 'New York', 'salary': 90000.0, 'start_date': '2023-08-20', 'email': 'charlie.brown@princeton.edu'}
{'id': 36, 'name': 'Bob Smith', 'age': 35, 'department': 'Sales', 'city': 'Austin', 'salary': 85000.0, 'start_date': '2023-07-15', 'email': 'bob.smith@princeton.edu'}
{'id': 35, 'name': 'Alice Johnson', 'age': 28, 'department': 'Engineering', 'city': 'Chicago', 'salary': 95000.0, 'start_date': '2023-06-01', 'email': 'alice.johnson@princeton.edu'}
{'id': 34, 'name': 'Chad Palumbo', 'age': 23, 'department': 'Marketing', 'city': 'New York', 'salary': 110000.0, 'start_date': '2026-01-15', 'email': 'chad.palumbo@princeton.edu'}
{'id': 25, 'name': 'Zoe Williams', 'age': 23, 'department': 'Marketing', 'city': 'San Francisco', 'salary': 54000.0, 'start_date': '2023-08-15', 'email': 'zoe.williams@acmecorp.com'}


---
## Task 4 — UPDATE Records
**15 points**

Modify existing records in the database.

**Requirements:**
1. Give **Alice Nguyen** a raise — update her salary to `$102,000`.
2. Apply a **5% raise** to every employee in the `Engineering` department.
3. Update the `city` of **Hana Sato** from `Chicago` to `Austin`.
4. After each update, run a SELECT to verify the change took effect and print the result.

> 💡 **Hint:** For the percentage raise, you can do the math directly in SQL: `SET salary = salary * 1.05`  — SQLite will calculate the new value for every matching row.

In [4]:
# Task 4 — UPDATE records
conn = sqlite3.connect("company.db")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

# 1. Give Alice a raise to $102,000
cursor.execute('''UPDATE employees
               SET salary = ?
               WHERE name = ?''',
               (102000, "Alice Nguyen"))

# Verify
cursor.execute("SELECT name, salary FROM employees WHERE name = ?", ("Alice Nguyen",))
row = cursor.fetchone()
print(f"Alice's new salary: ${row['salary']:,.2f}")

# 2. 5% raise for all Engineering employees
cursor.execute('''UPDATE employees
               SET salary = salary * 1.05
               WHERE department = ?''',
               ("Engineering",))

# Verify — print all Engineering salaries
print("\n--- Engineering salaries after raise ---")
cursor.execute("SELECT name, salary FROM employees WHERE department = ? ORDER BY salary DESC", ("Engineering",))
for row in cursor.fetchall():
    print(f"  {row['name']:<20} ${row['salary']:>10,.2f}")

# 3. Move Hana Sato to Austin
cursor.execute('''UPDATE employees
               SET city = ?
               WHERE name =?''',
               ("Austin", "Hana Sato"))

# Verify
cursor.execute("SELECT name, city FROM employees WHERE name = ?", ("Hana Sato",))
print(f"\nHana Sato's new city: {cursor.fetchone()['city']}")

conn.commit()
conn.close()

Alice's new salary: $102,000.00

--- Engineering salaries after raise ---
  Victor Alves         $144,703.12
  Marcus Reed          $137,757.38
  Carol Kim            $129,654.00
  Rachel Wong          $123,865.88
  Grace Thompson       $119,235.38
  Nina Johansson       $114,604.88
  Ivan Rossi           $113,447.25
  Yuki Tanaka          $108,816.75
  Alice Nguyen         $107,100.00
  Alice Johnson        $ 99,750.00

Hana Sato's new city: Austin


---
## Task 5 — DELETE Records
**10 points**

Remove records from the database safely.

**Requirements:**
1. Delete the three employees you inserted in Task 3 using a single `DELETE` statement with `WHERE id > ?` — pass in the id of the last original employee (25) as the threshold.
2. Print the employee count before and after the deletion to confirm.
3. Verify the original 25 employees are all still present by printing a count.

> ⚠️ **Warning:** A `DELETE` without a `WHERE` clause deletes **every row** in the table. Always double-check your `WHERE` condition before running a delete.

In [5]:
# Task 5 — DELETE records
conn = sqlite3.connect("company.db")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

# Count before
cursor.execute("SELECT COUNT(*) AS total FROM employees")
before = cursor.fetchone()["total"]
print(f"Count before delete: {before}")

# 1. Delete the employees added in Task 3 (id > 25)
cursor.execute('''DELETE FROM employees
               WHERE id > ?''',
               (25,))
conn.commit()

# Count after
cursor.execute("SELECT COUNT(*) AS total FROM employees")
after = cursor.fetchone()["total"]
print(f"Count after delete:  {after}")
print(f"Rows removed: {before - after}")

conn.close()

Count before delete: 29
Count after delete:  25
Rows removed: 4


---
## Task 6 — Analytical Queries
**15 points**

Use SELECT with aggregate functions and GROUP BY to answer business questions.

**Requirements:**
1. Print the **number of employees per department**, ordered by headcount descending.
2. Print the **average salary per city**, formatted with `$` and commas, ordered highest to lowest.
3. Print the **top 5 highest-paid employees** — name, department, and salary.
4. Print the **number of employees per city** who earn more than the overall average salary.  
   > Hint: Use a subquery — `WHERE salary > (SELECT AVG(salary) FROM employees)`

> 💡 **Hint for GROUP BY:** `SELECT department, COUNT(*) AS headcount FROM employees GROUP BY department` groups all rows by department and counts each group.

In [6]:
# Task 6 — Analytical queries
conn = sqlite3.connect("company.db")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

# 1. Headcount per department
print("--- Headcount by Department ---")
cursor.execute('SELECT department, COUNT(*) AS count FROM employees GROUP BY department ORDER BY count DESC')
for row in cursor.fetchall():
    print(f"  {row['department']:<20} {row['count']:>5}")

# 2. Average salary by city
print("\n--- Average Salary by City ---")
cursor.execute('SELECT city, AVG(salary) AS avg_salary FROM employees GROUP BY city ORDER BY avg_salary DESC')
for row in cursor.fetchall():
    print(f"  {row['city']:<20} ${row['avg_salary']:>10,.2f}")

# 3. Top 5 earners
print("\n--- Top 5 Earners ---")
cursor.execute('SELECT name, salary FROM employees ORDER BY salary DESC LIMIT 5')
for row in cursor.fetchall():
    print(f"  {row['name']:<20} ${row['salary']:>10,.2f}")

# 4. Employees above average salary, grouped by city
print("\n--- Above-Average Earners per City ---")
cursor.execute('SELECT name, city, salary FROM employees WHERE salary > (SELECT AVG(salary) FROM employees) ORDER BY city')
for row in cursor.fetchall():
    print(f"  {row['name']:<20} {row['city']:<20} ${row['salary']:>10,.2f}")

conn.close()

--- Headcount by Department ---
  Engineering              9
  Sales                    6
  Marketing                6
  HR                       4

--- Average Salary by City ---
  San Francisco        $104,316.36
  Austin               $ 89,664.77
  New York             $ 75,166.67
  Chicago              $ 70,163.35

--- Top 5 Earners ---
  Victor Alves         $144,703.12
  Marcus Reed          $137,757.38
  Carol Kim            $129,654.00
  Rachel Wong          $123,865.88
  Grace Thompson       $119,235.38

--- Above-Average Earners per City ---
  Grace Thompson       Austin               $119,235.38
  Ivan Rossi           Austin               $113,447.25
  Nina Johansson       Austin               $114,604.88
  Rachel Wong          Austin               $123,865.88
  Yuki Tanaka          Chicago              $108,816.75
  Paula Diaz           New York             $ 87,500.00
  Alice Nguyen         San Francisco        $107,100.00
  Carol Kim            San Francisco        $129,6

---
## Task 7 — Import CSV into the Database
**10 points**

Combine everything from Weeks 2 and 3: read your cleaned CSV and load it into a fresh table.

**Requirements:**
1. Copy `employees_clean.csv` from your `week2/` folder into this `week3/` folder.
2. Create a new table called `employees_imported` with the same columns as `employees` (minus `id` — let SQLite auto-assign that).
3. Loop through `employees_clean.csv` using `csv.DictReader` and insert each row into `employees_imported` using a parameterized query.
4. After importing, run a SELECT to print the total count and the first 5 rows of `employees_imported`.

> 💡 **Hint:** Some rows may have `age` as `None` (from your Week 2 cleaning). SQLite accepts `None` as `NULL` — no special handling needed.

In [7]:
# Task 7 — Import CSV into the database
import csv

conn = sqlite3.connect("company.db")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

# 2. Create the employees_imported table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS employees_imported (
        id         INTEGER PRIMARY KEY AUTOINCREMENT,
        name       TEXT    NOT NULL,
        age        INTEGER,
        department TEXT,
        city       TEXT,
        salary     REAL    NOT NULL,
        start_date TEXT,
        email      TEXT
    )
""")

# 3. Read CSV and insert rows
imported = 0
try:
    with open("employees_clean.csv", "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            # Convert types — age may be blank
            age    = int(row["age"]) if row["age"] else None
            salary = float(row["salary"])
            cursor.execute(
                "INSERT INTO employees_imported (name, age, department, city, salary, start_date, email) VALUES (?, ?, ?, ?, ?, ?, ?)",
                (row["name"], age, row["department"], row["city"], salary, row["start_date"], row["email"])
            )
            imported += 1
    conn.commit()
    print(f"Imported {imported} rows into employees_imported.")
except FileNotFoundError:
    print("employees_clean.csv not found — copy it from your week2/ folder first.")

# 4. Verify
print("\n--- First 5 imported rows ---")
cursor.execute("SELECT id, name, department, city, salary FROM employees_imported LIMIT 5")
for row in cursor.fetchall():
    print(f"  {row['id']:>3}  {row['name']:<20} {row['department']:<14} {row['city']:<15} ${row['salary']:,.2f}")

conn.close()

Imported 25 rows into employees_imported.

--- First 5 imported rows ---
    1  Alice Nguyen         Engineering    San Francisco   $95,000.00
    2  Bob Martinez         Marketing      Chicago         $62,000.00
    3  Carol Kim            Engineering    San Francisco   $112,000.00
    4  Dave Okafor          Sales          New York        $71,000.00
    5  Eve Patel            Marketing      Chicago         $58,000.00


---
## Task 8 — Commit to GitHub
**10 points (split from above)**

Save and push all your Week 3 files.

**Steps:**
1. Save this notebook: **File → Save Notebook** (`Cmd+S`).
2. Make sure these files are all in your `week3/` folder:
   - `week3_assignment.ipynb`
   - `company.db`
   - `employees_clean.csv`
3. Open Terminal and run:

```bash
cd ~/Documents/python-course
git add .
git commit -m "Week 3 assignment: SQLite database and queries"
git push
```

---
## Grading Rubric

| Task | Points | Full marks when… |
|------|--------|-------------------|
| 1 — Explore | 15 | Tables, departments, count, and columns all print correctly |
| 2 — SELECT | 20 | All 5 queries return correct results; output formatted neatly |
| 3 — INSERT | 15 | 4 rows added; count confirmed; `lastrowid` used |
| 4 — UPDATE | 15 | All 3 updates applied; each verified with a SELECT |
| 5 — DELETE | 10 | Task 3 rows removed; count before/after printed |
| 6 — Analytics | 15 | All 4 queries correct; subquery used in requirement 4 |
| 7 — CSV Import | 10 | `employees_imported` table created and populated |
| **Total** | **100** | |

---
## Before You Submit — Checklist

- [ ] All cells run top-to-bottom without errors (**Kernel → Restart & Run All**)
- [ ] `row_factory = sqlite3.Row` is set in every task
- [ ] All INSERT / UPDATE / DELETE operations are followed by `conn.commit()`
- [ ] All queries use `?` placeholders — no f-strings inside SQL strings
- [ ] `employees_imported` table exists and has the correct row count
- [ ] All three files committed to `week3/` on GitHub

In [9]:
# Self-check — DO NOT EDIT THIS CELL
import sqlite3

print("=" * 45)
print("  Week 3 Self-check")
print("=" * 45)

checks = []

conn = sqlite3.connect("company.db")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

# Check 1: employees table has the original 25 rows
try:
    cursor.execute("SELECT COUNT(*) AS n FROM employees")
    count = cursor.fetchone()["n"]
    assert count == 25, f"Expected 25 employees, found {count}"
    checks.append(("employees table has 25 rows", True, ""))
except Exception as e:
    checks.append(("employees table has 25 rows", False, str(e)))

# Check 2: departments table has 4 rows
try:
    cursor.execute("SELECT COUNT(*) AS n FROM departments")
    count = cursor.fetchone()["n"]
    assert count == 4, f"Expected 4 departments, found {count}"
    checks.append(("departments table has 4 rows", True, ""))
except Exception as e:
    checks.append(("departments table has 4 rows", False, str(e)))

# Check 3: Alice's salary was updated
try:
    cursor.execute("SELECT salary FROM employees WHERE name = ?", ("Alice Nguyen",))
    salary = cursor.fetchone()["salary"]
    assert salary == 107100.0, f"Expected Alice's salary to be 107100, got {salary}"
    checks.append(("Alice's salary updated to $107,100", True, ""))
except Exception as e:
    checks.append(("Alice's salary updated to $107,100", False, str(e)))

# Check 4: Hana Sato's city was updated
try:
    cursor.execute("SELECT city FROM employees WHERE name = ?", ("Hana Sato",))
    city = cursor.fetchone()["city"]
    assert city == "Austin", f"Expected Hana's city to be Austin, got {city}"
    checks.append(("Hana Sato moved to Austin", True, ""))
except Exception as e:
    checks.append(("Hana Sato moved to Austin", False, str(e)))

# Check 5: employees_imported table exists and has rows
try:
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='employees_imported'")
    assert cursor.fetchone() is not None, "employees_imported table does not exist"
    cursor.execute("SELECT COUNT(*) AS n FROM employees_imported")
    count = cursor.fetchone()["n"]
    assert count > 0, "employees_imported is empty"
    checks.append((f"employees_imported has {count} rows", True, ""))
except Exception as e:
    checks.append(("employees_imported table populated", False, str(e)))

# Check 6: no Task 3 inserts still present (id > 25 should be gone)
try:
    cursor.execute("SELECT COUNT(*) AS n FROM employees WHERE id > 25")
    count = cursor.fetchone()["n"]
    assert count == 0, f"{count} Task 3 inserts still present — did you run Task 5?"
    checks.append(("Task 3 inserts cleaned up", True, ""))
except Exception as e:
    checks.append(("Task 3 inserts cleaned up", False, str(e)))

conn.close()

# Print results
passed = 0
for name, ok, msg in checks:
    icon   = "✓" if ok else "✗"
    status = "PASS" if ok else "FAIL"
    print(f"  {icon} {name}: {status}")
    if msg:
        print(f"      → {msg}")
    if ok:
        passed += 1

print("=" * 45)
print(f"  {passed}/{len(checks)} checks passed")
if passed == len(checks):
    print("  All good — ready to push to GitHub!")
else:
    print("  Fix the issues above and run again.")
print("=" * 45)

  Week 3 Self-check
  ✓ employees table has 25 rows: PASS
  ✓ departments table has 4 rows: PASS
  ✓ Alice's salary updated to $107,100: PASS
  ✓ Hana Sato moved to Austin: PASS
  ✓ employees_imported has 75 rows: PASS
  ✓ Task 3 inserts cleaned up: PASS
  6/6 checks passed
  All good — ready to push to GitHub!
